# Local Image Classification Notebook

This notebook has been adapted from the Google Colab version to work with your local dataset. Follow these steps to run the notebook:

1. Ensure you have your dataset organized in the following structure inside the `ML FLASK/dataset` folder:
   ```
   dataset/
   ├── Garbage/
   │   ├── image1.jpg
   │   ├── image2.jpg
   │   └── ...
   ├── manhole/
   │   ├── image1.jpg
   │   └── ...
   └── pothole/
       ├── image1.jpg
       └── ...
   ```

2. Run each cell in order, starting from the top.
3. The model will be trained and saved in the `ML FLASK/models` directory.
4. The best model will be copied to `ML FLASK/best_model.h5` for easy use with the Flask app.

**Required packages:**
- tensorflow
- matplotlib
- split-folders
- numpy
- tkinter (for file selection)
- opencv-python (for the Flask app)

In [18]:
# Install required packages if needed
# Uncomment and run this cell if you're missing any packages

import sys
import subprocess

def install_package(package):
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print(f"✅ {package} installed successfully")

# packages = ["tensorflow", "matplotlib", "split-folders", "opencv-python"]
# for package in packages:
#     try:
#         __import__(package.replace("-", "_"))
#         print(f"✓ {package} already installed")
#     except ImportError:
#         install_package(package)

In [19]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPU available: []


In [20]:
import os
print("Working directory:", os.getcwd())
# You don't need to mount Google Drive when working locally

Working directory: d:\FINAL YEAR PROJECT\FINAL\ML FLASK


In [21]:
%pip uninstall -y split-folders
%pip install split-folders --upgrade


Found existing installation: split-folders 0.5.1
Uninstalling split-folders-0.5.1:
  Successfully uninstalled split-folders-0.5.1
Note: you may need to restart the kernel to use updated packages.
  Using cached split_folders-0.5.1-py3-none-any.whl.metadata (6.2 kB)
Using cached split_folders-0.5.1-py3-none-any.whl (8.4 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached split_folders-0.5.1-py3-none-any.whl.metadata (6.2 kB)
Using cached split_folders-0.5.1-py3-none-any.whl (8.4 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import splitfolders
import os
import shutil

# We will use data_path that is defined in the next cell
# Note: This cell should be run AFTER the data_path is defined

chnage name from val to test


In [23]:
import os
import shutil

old_path = os.path.join(os.getcwd(), "split_data", "val")
new_path = os.path.join(os.getcwd(), "split_data", "test")

if os.path.exists(old_path):
    shutil.move(old_path, new_path)
    print("✅ Renamed 'val' → 'test'")
else:
    print("❌ 'val' folder not found")

❌ 'val' folder not found


In [24]:
import os

# The dataset folders (Garbage, manhole, pothole) are directly in the ML FLASK directory
data_path = os.getcwd()  # This points directly to ML FLASK where our folders are
print(f"Dataset path: {data_path}")

# Example of what your folder structure should look like:
# D:\FINAL YEAR PROJECT\FINAL\ML FLASK\garbage\image1.jpg
# D:\FINAL YEAR PROJECT\FINAL\ML FLASK\manhole\image1.jpg
# D:\FINAL YEAR PROJECT\FINAL\ML FLASK\pothole\image1.jpg

# Check if the class directories exist
expected_classes = ['garbage', 'manhole', 'pothole']
missing_classes = []

for class_name in expected_classes:
    class_path = os.path.join(data_path, class_name)
    if os.path.exists(class_path):
        print(f"✅ Found {class_name} directory")
    else:
        missing_classes.append(class_name)
        print(f"❌ Missing {class_name} directory")

if missing_classes:
    print("\nWarning: Some class directories are missing!")
else:
    print("\n✅ All class directories found!")

# Create output directory for split data
split_data_path = os.path.join(os.getcwd(), "split_data")
os.makedirs(split_data_path, exist_ok=True)
print(f"\nSplit data will be saved to: {split_data_path}")

Dataset path: d:\FINAL YEAR PROJECT\FINAL\ML FLASK
✅ Found garbage directory
✅ Found manhole directory
✅ Found pothole directory

✅ All class directories found!

Split data will be saved to: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\split_data


In [25]:
import splitfolders
import os
import shutil

# Clean previous split if any
if os.path.exists(split_data_path):
    print("Cleaning previous split data...")
    shutil.rmtree(split_data_path)

# Create temporary dataset directory
temp_dataset = os.path.join(os.getcwd(), "temp_dataset")
if os.path.exists(temp_dataset):
    shutil.rmtree(temp_dataset)
os.makedirs(temp_dataset)

# Copy and flatten the directory structure
for class_name in ['garbage', 'manhole', 'pothole']:
    # Create class directory in temp_dataset
    class_dir = os.path.join(temp_dataset, class_name)
    os.makedirs(class_dir)
    
    # Source directory
    src_dir = os.path.join(os.getcwd(), class_name)
    if os.path.exists(src_dir):
        # Walk through all subdirectories
        for root, dirs, files in os.walk(src_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    src_file = os.path.join(root, file)
                    dst_file = os.path.join(class_dir, file)
                    shutil.copy2(src_file, dst_file)
        print(f"✅ Processed {class_name} images")

# Split the dataset
print("\nSplitting dataset...")
splitfolders.ratio(
    temp_dataset,
    output=split_data_path,
    seed=42,
    ratio=(0.8, 0.2),   # 80% train, 20% test
    move=False
)

# Clean up temporary directory
shutil.rmtree(temp_dataset)
print(f"✅ Dataset split into: {split_data_path}")

# Rename val to test if needed
old_path = os.path.join(split_data_path, "val")
new_path = os.path.join(split_data_path, "test")

if os.path.exists(old_path):
    shutil.move(old_path, new_path)
    print("✅ Renamed 'val' → 'test'")

Cleaning previous split data...
✅ Processed garbage images
✅ Processed garbage images
✅ Processed manhole images
✅ Processed manhole images
✅ Processed pothole images

Splitting dataset...
✅ Processed pothole images

Splitting dataset...
✅ Dataset split into: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\split_data
✅ Renamed 'val' → 'test'
✅ Dataset split into: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\split_data
✅ Renamed 'val' → 'test'


In [26]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

img_height, img_width = 224, 224
batch_size = 32

# Set paths
train_path = os.path.join(os.getcwd(), "split_data", "train")
test_path = os.path.join(os.getcwd(), "split_data", "test")

# Train + validation
train_val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2  # 20% of 80% = 16% val
)

train_gen = train_val_datagen.flow_from_directory(
    train_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = train_val_datagen.flow_from_directory(
    train_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Unseen test data (20%)
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    test_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print("✅ Data generators created successfully")
print(f"Classes: {train_gen.class_indices}")

Found 515 images belonging to 3 classes.
Found 128 images belonging to 3 classes.
Found 128 images belonging to 3 classes.
Found 161 images belonging to 3 classes.
✅ Data generators created successfully
Classes: {'garbage': 0, 'manhole': 1, 'pothole': 2}
Found 161 images belonging to 3 classes.
✅ Data generators created successfully
Classes: {'garbage': 0, 'manhole': 1, 'pothole': 2}


In [27]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')  # 3 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [28]:
from tensorflow.keras.callbacks import ModelCheckpoint
import os

# Save models in local 'models' directory
save_dir = os.path.join(os.getcwd(), "models")
os.makedirs(save_dir, exist_ok=True)

checkpoint_cb = ModelCheckpoint(
    filepath=os.path.join(save_dir, "epoch_{epoch:02d}_valacc_{val_accuracy:.2f}.h5"),
    monitor='val_accuracy',
    save_best_only=False,
    save_freq='epoch',
    verbose=1
)

print(f"✅ Model checkpoints will be saved to: {save_dir}")

✅ Model checkpoints will be saved to: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models


In [29]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[checkpoint_cb]
)


d:\FINAL YEAR PROJECT\FINAL\.venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 889ms/step - accuracy: 0.7715 - loss: 0.5573
Epoch 1: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_01_valacc_0.73.h5

Epoch 1: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_01_valacc_0.73.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.8990 - loss: 0.2665 - val_accuracy: 0.7344 - val_loss: 0.7792
Epoch 2/10
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step - accuracy: 0.9849 - loss: 0.0475
Epoch 2: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_02_valacc_0.71.h5

Epoch 2: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_02_valacc_0.71.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 630ms/step - accuracy: 0.9883 - loss: 0.0316 - val_accuracy: 0.7109 - val_loss: 1.0311
Epoch 3/10
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.9999 - loss: 0.0069
Epoch 3: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_03_valacc_0.77.h5

Epoch 3: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_03_valacc_0.77.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 632ms/step - accuracy: 0.9981 - loss: 0.0158 - val_accuracy: 0.7656 - val_loss: 0.9056
Epoch 4/10
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 459ms/step - accuracy: 1.0000 - loss: 0.0037
Epoch 4: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_04_valacc_0.80.h5

Epoch 4: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_04_valacc_0.80.h5


...

17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 593ms/step - accuracy: 1.0000 - loss: 0.0024 - val_accuracy: 0.7578 - val_loss: 0.9360
Epoch 7/10
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 473ms/step - accuracy: 1.0000 - loss: 0.0015
Epoch 7: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_07_valacc_0.74.h5

Epoch 7: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_07_valacc_0.74.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 624ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.7422 - val_loss: 1.0092
Epoch 8/10
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 494ms/step - accuracy: 1.0000 - loss: 0.0011
Epoch 8: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_08_valacc_0.73.h5

Epoch 8: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_08_valacc_0.73.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 632ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 0.7266 - val_loss: 1.0705
Epoch 9/10
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 457ms/step - accuracy: 1.0000 - loss: 6.3313e-04
Epoch 9: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_09_valacc_0.74.h5

Epoch 9: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_09_valacc_0.74.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 20s 586ms/step - accuracy: 1.0000 - loss: 7.3418e-04 - val_accuracy: 0.7422 - val_loss: 1.0377
Epoch 10/10
Epoch 10/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step - accuracy: 1.0000 - loss: 5.6576e-04
Epoch 10: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_10_valacc_0.72.h5

Epoch 10: saving model to d:\FINAL YEAR PROJECT\FINAL\ML FLASK\models\epoch_10_valacc_0.72.h5


17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 567ms/step - accuracy: 1.0000 - loss: 7.1471e-04 - val_accuracy: 0.7188 - val_loss: 1.0983


In [34]:
from tensorflow.keras.models import load_model
import glob
import os

# Path to saved models
save_dir = os.path.join(os.getcwd(), "models")
model_paths = sorted(glob.glob(os.path.join(save_dir, "*.h5")))

print(f"\n🧠 Found {len(model_paths)} saved models.\n")

for path in model_paths:
    loaded_model = load_model(path)
    loss, acc = loaded_model.evaluate(test_gen, verbose=0)
    print(f"{os.path.basename(path)} → Test Accuracy: {acc:.2%}")
    
# Select the best model based on validation accuracy
best_model_path = None
best_acc = 0

for path in model_paths:
    if "valacc" in path:
        acc_str = path.split("valacc_")[1].split(".h5")[0]
        try:
            acc = float(acc_str)
            if acc > best_acc:
                best_acc = acc
                best_model_path = path
        except:
            pass

if best_model_path:
    print(f"\n✅ Best model: {os.path.basename(best_model_path)} with val accuracy: {best_acc:.2%}")
    # Copy this model to a simpler name for Flask app
    import shutil
    target_path = os.path.join(os.getcwd(), "best_model.h5")
    shutil.copy(best_model_path, target_path)
    print(f"✅ Best model copied to: {target_path}")


🧠 Found 10 saved models.



epoch_01_valacc_0.73.h5 → Test Accuracy: 93.79%


epoch_02_valacc_0.71.h5 → Test Accuracy: 93.17%


epoch_03_valacc_0.77.h5 → Test Accuracy: 94.41%


epoch_04_valacc_0.80.h5 → Test Accuracy: 95.03%


...

epoch_06_valacc_0.76.h5 → Test Accuracy: 92.55%


epoch_07_valacc_0.74.h5 → Test Accuracy: 92.55%


epoch_08_valacc_0.73.h5 → Test Accuracy: 92.55%


epoch_09_valacc_0.74.h5 → Test Accuracy: 93.17%


epoch_10_valacc_0.72.h5 → Test Accuracy: 92.55%

✅ Best model: epoch_04_valacc_0.80.h5 with val accuracy: 80.00%
✅ Best model copied to: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\best_model.h5


CHECK ON RANDOM IMAGE--------------------------------------

UPLOAD THE IMAGE

In [31]:
import os
from tkinter import Tk, filedialog

def select_image():
    root = Tk()
    root.withdraw()  # Hide the main window
    file_path = filedialog.askopenfilename(
        title="Select Image",
        filetypes=[("Image files", "*.jpg *.jpeg *.png")]
    )
    root.destroy()
    return file_path

# Select a test image
img_path = select_image()
if img_path:
    print(f"Selected image: {img_path}")
else:
    print("No image selected. Using a sample from test directory instead.")
    # Get a sample image from the test directory
    test_path = os.path.join(os.getcwd(), "split_data", "test")
    for root, dirs, files in os.walk(test_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                print(f"Using sample image: {img_path}")
                break
        if img_path:
            break

No image selected. Using a sample from test directory instead.
Using sample image: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\split_data\test\garbage\010_jpg.rf.594589a565b1e48d3b13e1110676ebf8.jpg


LOAD THE MODEL , CHANGE PATH

In [32]:
from tensorflow.keras.models import load_model
import os

# Try to load the best model first
best_model_path = os.path.join(os.getcwd(), "best_model.h5")
if os.path.exists(best_model_path):
    model = load_model(best_model_path)
    print(f"✅ Best model loaded from: {best_model_path}")
else:
    # Otherwise try to find any model in the models directory
    model_dir = os.path.join(os.getcwd(), "models")
    if os.path.exists(model_dir):
        model_files = [f for f in os.listdir(model_dir) if f.endswith('.h5')]
        if model_files:
            model_path = os.path.join(model_dir, model_files[0])
            model = load_model(model_path)
            print(f"✅ Model loaded from: {model_path}")
        else:
            print("❌ No model files found in the models directory")
    else:
        print("❌ Models directory not found")

✅ Best model loaded from: d:\FINAL YEAR PROJECT\FINAL\ML FLASK\best_model.h5


In [33]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Define the class labels (alphabetical if using flow_from_directory)
class_labels = ['Garbage', 'manhole', 'pothole']  # Adjust order if needed

# Check if img_path is defined and exists
if 'img_path' in locals() and os.path.exists(img_path):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # (1, 224, 224, 3)

    # Show image
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Selected Image")
    plt.show()
else:
    print("❌ No valid image path found")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import numpy as np

# Check if the model and image array are available
if 'model' in locals() and 'img_array' in locals():
    try:
        pred = model.predict(img_array)
        predicted_index = np.argmax(pred)
        predicted_class = class_labels[predicted_index]

        print(f"🧠 Predicted class: {predicted_class}")
        print(f"🔢 Confidence: {pred[0][predicted_index]:.2%}")
        
        # Show the prediction on the image
        plt.figure(figsize=(6, 6))
        plt.imshow(image.array_to_img(img_array[0]))
        plt.title(f"Prediction: {predicted_class} ({pred[0][predicted_index]:.2%})")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"❌ Error during prediction: {str(e)}")
else:
    if 'model' not in locals():
        print("❌ Model not available")
    if 'img_array' not in locals():
        print("❌ Image not available")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
🧠 Predicted class: manhole
🔢 Confidence: 99.96%


In [ ]:
# Export model and classes for Flask app
import os
import json

if 'model' in locals():
    # Save model for Flask app if not already saved
    flask_model_path = os.path.join(os.getcwd(), "epoch_10_valacc_0.96.h5")
    if not os.path.exists(flask_model_path):
        model.save(flask_model_path)
        print(f"✅ Model saved for Flask app: {flask_model_path}")
    
    # Save class labels
    class_labels_path = os.path.join(os.getcwd(), "class_labels.json")
    with open(class_labels_path, 'w') as f:
        json.dump(class_labels, f)
    print(f"✅ Class labels saved to: {class_labels_path}")
    
    print("\nYour model is ready for the Flask API! You can now run app.py")
else:
    print("❌ Model not available, cannot export for Flask app")